In [1]:
# pipeline.py (ex-main.py, quasi inchangé, juste wrappé dans une fonction)
import os
import sys
from pathlib import Path
import pandas as pd


In [2]:
SRC_DIR = Path.cwd()
print(f"SRC_dir : {SRC_DIR}")
BASE_DIR = str(Path(SRC_DIR).resolve().parent)
print(BASE_DIR)

SRC_dir : c:\Users\Cyril\Documents\python\jedha\jedha_Full_stack\Jedha_certif_CDSD\Block1\Kayak_weather_hotels\src
C:\Users\Cyril\Documents\python\jedha\jedha_Full_stack\Jedha_certif_CDSD\Block1\Kayak_weather_hotels


In [3]:
import importlib, inspect
import get_weather_forecasts as gwf

print("Fichier chargé :", gwf.__file__)
print("Fonctions publiques :", [n for n, f in inspect.getmembers(gwf, inspect.isfunction)])

# Relit le fichier depuis le disque
gwf = importlib.reload(gwf)

https://nominatim.openstreetmap.org/search
Fichier chargé : c:\Users\Cyril\Documents\python\jedha\jedha_Full_stack\Jedha_certif_CDSD\Block1\Kayak_weather_hotels\src\get_weather_forecasts.py
Fonctions publiques : ['NamedTuple', 'compile_weather', 'compile_weather2', 'get_weather', 'get_weather_data_for_cities', 'run_weather_searches', 'select_best_weather_cities', 'summarize_weather']


In [4]:

import config_kayak
from config_kayak import (
    AWS_BUCKET_NAME, AWS_BUCKET_DIR, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY,
    DATA_DIR_CSV,
)
from load_cities import load_cities, get_coordinates_cities
from get_weather_forecasts import get_weather_data_for_cities, select_best_weather_cities, run_weather_searches

from scrap_hotels import collect_cities_hotels, select_top_hotels
from s3_utils import save_csv_to_s3, timestamped_filename, read_csv_from_s3
from rds_utils import get_rds_engine, save_df_to_rds, get_pg_engine

from availability import build_date_windows, run_hotel_searches, keys_from_df, DEFAULT_OCCUPANCY

CHECKIN_DATE_NR = 1
CHECKOUT_DATE_NR = -1



In [6]:
# SRC_DIR = Path(__file__).resolve().parent
#BASE_DIR = SRC_DIR.parent
os.chdir(BASE_DIR)


#get cities coord & save to s3
scraped_dt = pd.Timestamp.now()  # scraped_datetime added 01/10/2026
cities_list_df = load_cities()
results_df = get_coordinates_cities(cities_list_df)
results_df.to_csv(os.path.join(DATA_DIR_CSV, "cities.csv"), index=False, encoding="utf-8")
save_csv_to_s3(results_df, "cities.csv")

# get weather data on 5 days free formula 
df_cities = results_df
# scraped_dt
result_weather = get_weather_data_for_cities(df_cities, scraped_dt = scraped_dt)
result_weather['scraped_at'] = scraped_dt

# Weather summary on different time windows (for trip)
# build different time windows for weather and hotels search target
windows = build_date_windows(result_weather["date"].unique())

date_span = result_weather["date"].unique()
checkin_date = date_span[CHECKIN_DATE_NR].isoformat()
checkout_date = date_span[CHECKOUT_DATE_NR].isoformat()

# top_cities = select_best_weather_cities(
#     result_weather, top_n=5, start_date=checkin_date, end_date=checkout_date
# )
engine = get_pg_engine() #use engine with default DATABASE_URL default variable

   
top_cities = run_weather_searches(
               result_weather, windows,
               select_weather_cities_fn = select_best_weather_cities,
               scraped_dt = scraped_dt,
               top_n = 5,
               )
print(f"top cities before \n {top_cities}")

C:\Users\Cyril\AppData\Local\Temp\ipykernel_3568\735419816.py:8: UserWarning: More than 35 in cities file from C:\Users\Cyril\Documents\python\jedha\jedha_Full_stack\Jedha_certif_CDSD\Block1\Kayak_weather_hotels/data/config/cities.csv
  cities_list_df = load_cities()
INFO:botocore.credentials:Found credentials in environment variables.
INFO:s3_utils:CSV sauvegardé sur s3://kayak-cyril/kayak/cities.csv


select best weather cities for window 2026-10-02 - 2026-10-05
select best weather cities for window 2026-10-04 - 2026-10-05
top cities before 
                             city  avg_comfort_score checkin_date  \
0                    La Rochelle             0.6550   2026-10-02   
1                Aix en Provence             0.5525   2026-10-02   
2                          Paris             0.5475   2026-10-02   
3               Gorges du Verdon             0.5375   2026-10-02   
4                      Montauban             0.5350   2026-10-02   
..                           ...                ...          ...   
69                         Dijon             0.3650   2026-10-04   
70                    Strasbourg             0.3400   2026-10-04   
71                        Colmar             0.3200   2026-10-04   
72                     Eguisheim             0.3150   2026-10-04   
73  Chateau du Haut Koenigsbourg             0.3000   2026-10-04   

   checkout_date  rain_sum       temp  